**This system leverages representations from RAVE. Learn more about that project here:** 
https://github.com/acids-ircam/RAVE

**Download more generative models from:** 
https://acids-ircam.github.io/rave_models_download

**Train your own model using:** 
https://colab.research.google.com/drive/1ih-gv1iHEZNuGhHPvCHrleLNXvooQMvI?usp=sharing (thanks to https://github.com/moiseshorta !)

**Giving Feedback** 
https://forms.office.com/Pages/DesignPageV2.aspx?prevorigin=Marketing&origin=NeoPortalPage&subpage=design&id=kfCdVhOw40CG7r2cueJYFFaGvjbF1INKikqB9y_N2gRUODBTNTJKM09HVjlFQjFRUlQ5U05QUVQ5MS4u&topview=Prefill

In [7]:
# In[0] # Load Utility Functions:
from load_generative_model import Model
from IPython.display import Audio, display
# from gui import interface
import librosa as li
import torch


In [2]:
# Working with trajectories in latent audio models

# In[1] # Pick Model:
model_name: str = 'percussion'
model_location:str = 'generative_models/'+model_name+'.ts'

control_model_location = 'control_models/vae_scripted_model.ts'
model = Model([model_location])
sr: int =44100

In [15]:
import os
import random

# In[2] # Pick Audio Samples:
# Get all sound files from the 'sounds' folder
sound_files = [f for f in os.listdir('sounds') if f.endswith(('.wav', '.aif', '.mp3', '.ogg'))]

# Select 2 random sounds
selected_sounds = random.sample(sound_files, min(2, len(sound_files)))
print(f"Selected sounds:")

for sound in selected_sounds:
    print(f"- {sound}")
    audio_location = os.path.join('sounds', sound)
    audio, sr = li.load(audio_location,sr=44100)
    display(Audio(audio, rate=sr))

encodings = [model.encode(li.load(audio_location,sr=44100)[0]) for audio_location in [os.path.join('sounds', sound) for sound in selected_sounds]]
for i, enc in enumerate(encodings):
    print(f"Encoding {i} shape: {enc[0].shape}")
    audio_recon = model.decode(enc[0])
    display(Audio(audio_recon, rate=sr))



Selected sounds:
- Guiro C78 Short b.aif


- Rim C78.aif


Encoding 0 shape: (1, 4, 6)


Encoding 1 shape: (1, 4, 1)


In [16]:
print('encoding shapes:', [enc[0].shape for enc in encodings])

def resample_encodings(encodings):
    max_len = max(enc[0].shape[-1] for enc in encodings)
    resampled_encodings = [torch.nn.functional.interpolate(torch.from_numpy(enc[0]), size=max_len).numpy() for enc in encodings]
    return resampled_encodings

resampled_encodings = resample_encodings(encodings)
for i, enc in enumerate(resampled_encodings):
    print(f"Encoding {i} shape: {enc.shape}")
    audio_recon = model.decode(enc)
    display(Audio(audio_recon, rate=sr))


encoding shapes: [(1, 4, 6), (1, 4, 1)]
Encoding 0 shape: (1, 4, 6)


Encoding 1 shape: (1, 4, 6)


In [19]:
# Interpolate between encodings using a slider (works for any number of encodings)

import numpy as np
from ipywidgets import FloatSlider, interact, VBox, Label

# Number of interpolation steps (slider resolution)
steps = 100

# Create a slider for interpolation position (0 to len(encodings)-1)
slider = FloatSlider(min=0, max=len(resampled_encodings)-1, step=0.01, value=0, description='Interpolate')

# Interpolation function: linear between nearest encodings
# For n encodings, slider position x in [0, n-1]
def interpolate_and_play(x):
    left = int(np.floor(x))
    right = min(left+1, len(resampled_encodings)-1)
    alpha = x - left
    # Match the length of the longest encoding by resampling the shortest one, then interpolate
    left_enc = resampled_encodings[left]
    right_enc = resampled_encodings[right]  
    print(f"Left encoding shape: {left_enc.shape}, Right encoding shape: {right_enc.shape}")
    print(f"Left name: {selected_sounds[left]}, Right name: {selected_sounds[right]}")
    left_len = left_enc.shape[-2]
    right_len = right_enc.shape[-2]
    max_len = max(left_len, right_len)

    def resample_encoding(enc, target_len):
        # Linear interpolation along the length axis
        orig_len = enc.shape[-2]
        idx = np.linspace(0, orig_len - 1, target_len)
        # Assume enc shape (..., length, features)
        return np.stack([np.interp(idx, np.arange(orig_len), enc[..., i]) for i in range(enc.shape[-1])], axis=-1)

    if left_len != max_len:
        left_enc_rs = resample_encoding(left_enc.squeeze(), max_len)
    else:
        left_enc_rs = left_enc.squeeze()
    if right_len != max_len:
        right_enc_rs = resample_encoding(right_enc.squeeze(), max_len)
    else:
        right_enc_rs = right_enc.squeeze()

    interp = (1 - alpha) * left_enc_rs + alpha * right_enc_rs
    interp = interp[np.newaxis, ...]  # add batch dimension if needed
    enc_left = encodings[left][0]
    enc_right = encodings[right][0]
    interp = (1-alpha)*enc_left + alpha*enc_right
    audio_interp = model.decode(interp)
    print(f"Interpolating between {left} and {right} (alpha={alpha:.2f})")
    display(Audio(audio_interp, rate=sr))

interact(interpolate_and_play, x=slider)

print('use slider to interpolate between sounds')

interactive(children=(FloatSlider(value=0.0, description='Interpolate', max=1.0, step=0.01), Output()), _dom_c…

use slider to interpolate between sounds
